# 镜像中转搬运 notebook（本地 Docker Hub → 华为云 SWR cn-north-4）

> **何时需要**：仅当你的机器在**国外/国际线路**、要把镜像传进**国内** SWR（如北京 cn-north-4，
> 国际入中上行被限 ~20KB/s，docker push 全量层基本传不动）才走这条中转路。
> 机器在中国大陆的话**不需要本 notebook**：`docker login` + `docker push` 直推即可（GUIDE.md 阶段 5A 主路线）。
>
> 原理：本机先把镜像推到 Docker Hub（国际线路对国际枢纽快），再在**同区域的
> ModelArts CPU notebook** 里用 crane 把镜像从 Docker Hub 搬到 SWR（notebook 与
> SWR 同在华为云内网，实测双向 ~8MB/s 量级）。
>
> 使用前替换占位符：`<DOCKER_HUB_USER>` / `<DOCKER_HUB_TOKEN>` / `<SWR_LOGIN_USER>` / `<SWR_LOGIN_PASSWORD>` /
> `<IMAGE>` / `<TAG>` / `<SWR_ORG>`。登录指令在 SWR 控制台「客户端上传 → 登录指令」里整条复制。

**坑位速查**（都有实证）：
- GitHub 直连下载 crane 可能截断 → 下载命令带 ghproxy 兜底（cell 1）；
- 源镜像走 `dockerproxy.net` 镜像源（DaoCloud 官方源有白名单，会拒个人仓库）；
- 必须加 `--platform linux/amd64` 剥掉 buildx attestation index，否则 SWR 报 `MANIFEST_INVALID`
  （与本地构建带 `--provenance=false` 是同一个坑的两端）。


In [ ]:
import os

os.makedirs("/tmp/relay", exist_ok=True)
%cd /tmp/relay

# 下载 crane（go-containerregistry 的 CLI）。GitHub 直连可能截断，ghproxy 兜底。
!curl -sL -o crane.tar.gz   "https://github.com/google/go-containerregistry/releases/download/v0.20.3/go-containerregistry_Linux_x86_64.tar.gz"   || curl -sL -o crane.tar.gz   "https://mirror.ghproxy.com/https://github.com/google/go-containerregistry/releases/download/v0.20.3/go-containerregistry_Linux_x86_64.tar.gz"

!tar xzf crane.tar.gz crane
!./crane version

In [ ]:
# 登录两端 registry（密码走命令行参数，notebook 用完即弃；Docker Hub 用 access token 别用主密码）
# SWR 登录指令：SWR 控制台右上「客户端上传 → 登录指令」，user 形如 'cn-north-4@<账号域名>'
!./crane auth login docker.io -u <DOCKER_HUB_USER> -p <DOCKER_HUB_TOKEN>
!./crane auth login swr.cn-north-4.myhuaweicloud.com -u <SWR_LOGIN_USER> -p <SWR_LOGIN_PASSWORD>

In [ ]:
import subprocess, time

# 要搬的 tag（本链路通常就一个训练镜像）
TAGS = ["<TAG>"]
SRC_FMT = "dockerproxy.net/<DOCKER_HUB_USER>/<IMAGE>:{t}"
DST_FMT = "swr.cn-north-4.myhuaweicloud.com/<SWR_ORG>/<IMAGE>:{t}"

LOG = "/tmp/relay/crane_copy.log"
for TAG in TAGS:
    src, dst = SRC_FMT.format(t=TAG), DST_FMT.format(t=TAG)
    print(f"crane copy {src} -> {dst}")
    proc = subprocess.Popen(
        ["./crane", "copy", "-v", "--platform", "linux/amd64", src, dst],
        stdout=open(LOG, "a"), stderr=subprocess.STDOUT, cwd="/tmp/relay")
    t0 = time.time()
    rc = proc.wait()
    print(f"=== {TAG} 退出码 {rc}，耗时 {time.time()-t0:.0f}s ===", "DONE" if rc == 0 else "FAILED")
    os.system("tail -n 3 " + LOG)

print("
最后 6 行日志（每个 tag 应各有一条 digest 行，与本地 docker image inspect 对照）：")
os.system("tail -n 6 " + LOG)